In [ ]:
"""
Created By    : Jared W. Marquis
Creation Date : 01 August 2022
Course        : ATSC 528 - Atmospheric Data Analysis
Assignment    : #03 - Statistical Objective Analysis

Purpose:
Script to take sparse upper air observations and analyze them on a
polar stereographic map projection using statistical objective analysis.
[PUT MORE INFORMATION HERE - I.E., WHAT SPECIFIC THING IS BEING DONE]

"""
__author__    = "Jared W. Marquis"
__contact__   = "jared.marquis@und.edu"

In [ ]:
### Import Required Modules (shouldn't need to change) ###
import numpy as np                 #numpy for math
import matplotlib.pyplot as plt    #matplotlib for plotting
import cartopy.crs as ccrs         #cartopy for plotting on map
import cartopy.feature as cfeature #cartopy basic shapefiles

import pandas as pd # WOOOO PANDAAAAAS

In [ ]:
### Read in observations ###

# SID LAT LON HGT DIR SPD
# 0   1   2   3   4   5

# I think this should work on any OS now??? It works on my machine...
# it's importing a STANDARD LIBRARY MODULE so this should be fine and not count as me importing things im not allowed to
from pathlib import Path
observation_file = Path("RAOBs_201903131200.txt")

# This reads the data as comma-separated values, creating a pandas dataframe that is subsequently converted into a numpy array for easy manipulation later.
# Oh and it has no headers, which also makes things easier later.
observations = pd.read_csv(observation_file, header=None).to_numpy()

In [ ]:
### Set up analysis map with a 21x27 rectangular grid of points ###
''' We have a polar stereographic projection with a standard latitude of 60degN and a standard latitude of 100degW. Map scale is 1/15,000,000. 
    We need our origin at 90degN latitude and 0deg longitude, with grid points every 1.27cm extending to x = +- 25cm and y = +- 15.5cm.
    The first array member (0,0) is supposed to be at x = 25cm, y = -15.5cm.
'''

#np array element format: array[col][row], both starting at zero

# Init lists of x- & y-values
x = []
y = []

# x- & y-values at (0,0)
x0 = .25
y0 = -.155

# Populating the x & y lists with all the values on the grid
# You will note that all cm values have been converted to m
for i in range(27):
    diff = i * .0127
    if(i < 21):
        x.append(x0 + diff)
    y.append(y0 + diff)

x = np.array(x)
y = np.array(y)

# Use the x- & y-values to populate two new numpy arrays with the values necessary to actually graph all gridpoints
gridx, gridy = np.meshgrid(x,y)

testx = [-.25]
testy = [-.155]

# Plot the grid of values, to ensure that we made it correctly
fig, ax = plt.subplots()
ax.plot(gridy, -gridx, marker='o', color='k', linestyle='none')
ax.plot(testy, testx, marker='o', color='r', linestyle='none')
ax.grid()
plt.show()


In [ ]:
### convert obs lat/long to x,y ###

# Conversion factor we need cuz numpy trig fcns are in radians
degToRad = np.pi / 180

# Defining our standard latitude & longitude
standardLat = 60 * degToRad
standardLong = -100 * degToRad

# Earth's mean radius, in meters
rho = 6.371e6

# Map scale
m = 1 / 15e6

# Helper function for calculating scale factor
def __scaleFactor__(lat: float) -> float:
    sigmaNumerator = 1 + np.sin(standardLat)
    sigmaDenominator = 1 + np.sin(lat)

    return (sigmaNumerator / sigmaDenominator)

# Defining a function to convert from lat/long to x/y
def getCartesian(latDeg: float, longDeg: float) -> (float,float):
    lat = latDeg * degToRad
    long = (longDeg * degToRad) - standardLong
    sigma = __scaleFactor__(lat)
    r = m * rho * sigma * np.cos(lat)
    x = r * np.cos(long)
    y = r * np.sin(long)

    return x,y



# Initializing lists for the x- & y-values we will calculate from the observation lat/long
obsx = []
obsy = []

for i in range(observations.shape[0]):
    tempx, tempy = getCartesian(observations[i][1], observations[i][2])
    obsx.append(tempx)
    obsy.append(tempy)

# Turning lists into np arrays because they're much nicer to work with
obsx = np.array(obsx, dtype=np.double)
obsy = np.array(obsy, dtype=np.double)

# Check points with a graph
fig, ax = plt.subplots()
ax.plot(gridy, -gridx, marker='o', color='k', linestyle='none')
ax.plot(obsy, -obsx, marker='o', color='r', linestyle='none')
plt.show()

# The points do appear to be in the correct locations

In [ ]:
### GENERATE BACKGROUND FIELD ###


roi = .127775892 # meters


# functions for bilinear interpolation, because we are generating a background field, so we need values at observation points
def toIndexSpace(x: np.ndarray or float, x0: float) -> np.ndarray or float:
    delta = .0127
    return ( (x - x0) / delta )

def bilinearInterpolate(xo: float, yo: float, xGrid: np.ndarray, yGrid: np.ndarray, analysisArr: np.ndarray) -> float:
    '''Take an observation point and calculate the analysis value there with bilinear interpolation using the nearest four analysis points. See cell below for equation.'''


    # Defining the values at the origin of index space
    x0 = .25
    y0 = -.155

    # Transform the grid of analysis points into index space
    xIndices = toIndexSpace(xGrid, x0)
    yIndices = toIndexSpace(yGrid, y0)

    # Transform the observation point into index space
    xoIndex = toIndexSpace(xo, x0)
    yoIndex = toIndexSpace(yo, y0)

    # Calculate the x- and y-indices of the four points surrounding the observation point
    x1 = int(np.floor(xoIndex))
    x2 = x1 + 1
    y1 = int(np.floor(yoIndex))
    y2 = y1 + 1

    # Use the indices to find the analysis values at the four analysis points
    tlVal = analysisArr[x1][y1]
    trVal = analysisArr[x2][y1]
    blVal = analysisArr[x1][y2]
    brVal = analysisArr[x2][y2]

    # Construct the component matrices of the matrix-form bilinear interpolation equation (see the cell below)
    matX = np.array([x2 - xoIndex, xoIndex - x1])
    matQ = np.array([[tlVal, blVal], [trVal, brVal]])
    matY = np.array([[y2 - yoIndex], [yoIndex - y1]])

    # Calculate the analysis value at the observation point with some matrix multiplications (and an index because it returns a 1x1 numpy array and I want a float !!!)
    output = (matX @ matQ @ matY)[0]
    
    return output

def isOutsideGrid(x: float, y: float) -> bool:
    '''Determine whether a point is inside or outside the bounds of the analysis grid. Returns True if the point is OUTSIDE the grid.'''
    return (x < .25) or (x > .504) or (y < -.155) or (y > .1752)

def backgroundFromAnalysis(analysis: np.ndarray, gridAnalysis: np.ndarray, prevBG: np.ndarray, roi: float) -> np.ndarray:
    '''Take a set of analysis values and use them to generate background values at each observation point'''
    background = []
    
    for i in range(len(obsx)):

        if(isOutsideGrid(obsx[i],obsy[i])):
            # If the observation point is outside the analysis grid, we interpolate with the Cressman function
            bgVal = analyze(obsx[i], obsy[i], gridx.flatten(), gridy.flatten(), gridAnalysis.T.flatten(), roi)

        else:
            bgVal = bilinearInterpolate(obsx[i], obsy[i], x, y, gridAnalysis)

        background.append(bgVal)
    background = np.array(background)
    
    return background



# Defining the weight function used for the background
def weightFunc(dist: float, roi) -> float:
    w = 0

    if (dist <= roi):
        Kd = .00108844524 # sq meters
        numerator = -(dist ** 2)
        exponent = numerator / Kd
        w = np.exp(exponent)
        
    return w

def analyze(xa: float, ya: float, xArr: np.ndarray, yArr: np.ndarray, obsArr: np.ndarray, roi: float) -> float:
    '''
    Do a single-pass Barnes analysis at point (xa, ya) based on input observations.
    '''
    obsArr = np.asarray(obsArr, dtype=float)
    # Make arrays of the (signed) x- & y-distance between the analysis point and the observation points in the RoI
    xDist = xa - xArr
    yDist = ya - yArr
    
    # Initialize an empty np array that will be filled with the weights
    weights = np.ndarray((obsArr.shape[0],))

    # Populate the weights array using the weight function
    for i in range(obsArr.shape[0]):
        dist = np.hypot(xDist[i], yDist[i])
        weights[i] = weightFunc(dist, roi)

    # Initializing a couple variables that get their real value in a conditional later
    num = np.nansum(weights * obsArr)
    # Define the denominator of the fraction in the analysis function
    denom = np.nansum(weights)
    if (denom > 0):
        return num / denom
    else:
        return np.nan
    

bgAnalysis = []
bgGrid = np.ndarray((len(x),len(y)))

for i in range(len(x)):
    for j in range(len(y)):
        currVal = analyze(x[i], y[j], obsx, obsy, observations[:,3], roi)
        bgAnalysis.append(currVal)
        bgGrid[i][j] = currVal

bgAnalysis = np.array(bgAnalysis)

bgObservations = backgroundFromAnalysis(bgAnalysis, bgGrid, observations[:,3], roi)


The analysis equation we use for generating the background *and* for SOA is given by
$$
f_{A}(\vec{r}_{i}) = f_{B}(\vec{r}_{i}) + \sum_{k=1}^{K} w_{ik} [f_{O}(\vec{r}_{k}) - f_{B}(\vec{r}_{k})]
.$$
The difference is that we use different weight functions for each. The background uses
$$
w(d_{ik}) = \exp\left( \frac{-d^{2}_{ik}}{K_{d}} \right),
$$
while the actual analysis uses
$$
\sum_{l=1}^{K}w_{ik}\rho_{kl} = \rho_{ik.}
$$

I will not go into the details of what the constants are and how these weights are found, as those details are in the assignment paper.

Schlatter (1975) provides a formula for the great-circle distance $s$ between two points ${}(\lambda_{i},\varphi_{i}){}$ and ${}(\lambda_{j}, \varphi_{j}){}$:
$$
s = a \left[ (\lambda_{i} - \lambda_{j})^{2} \cos ^{2}\left( \frac{\phi_{i} + \phi_{j}}{2} \right) + (\phi_{i} - \phi_{j})^{2} \right]^{1/2}, 
$$
where $a$ is the Earth's radius in thousands of kms, and LAT AND LONG ARE IN RADIANS??? I mean, it makes sense for the cosine, but man that is weird.

We have the observation point lat/long, as those were provided in the dataset. However, we do not have analysis point lat/long, so those must be calculated.

In [ ]:
### ANALYSIS POINT LAT/LONG ###

def getLatLong(x: float, y: float) -> (float, float):
    ''' Calculate the latitude and longitude of a point (x,y), so that we can use the great circle formula to find distances between points '''
    x = x / m
    y = y / m

    latTanArg = np.sqrt((x ** 2) + (y ** 2)) / (rho * (1 + np.sin(standardLat)))
    lat = (np.pi / 2) - (2 * np.atan(latTanArg))

    long = np.atan2(y, x)

    lat = np.rad2deg(lat)
    long = np.rad2deg(long) + np.rad2deg(standardLong)

    return lat, long

# We put the lat/long in separate 2D arrays because it will make our lives much easier later
# That's because they'll have the same structure as some other arrays we'll be making
analysisLat = np.ndarray((len(x), len(y)))
analysisLong = np.ndarray((len(x), len(y)))

for i in range(len(x)):
    for j in range(len(y)):
        analysisLat[i,j], analysisLong[i,j] = getLatLong(x[i], y[j])

analysisLat = np.array(analysisLat)
analysisLong = np.array(analysisLong)


In [ ]:
### CALCULATE COVARIANCES ###



def greatCircleDistance(lat1: float, long1: float, lat2: float, long2: float) -> float:
    '''Calculate the great-circle distance (in thousands of kms) between two points, which are taken to be in lat/long degrees'''

    # We need the lat/long in radians because we are in crazy town
    lat1 = np.deg2rad(lat1)
    long1 = np.deg2rad(long1)
    lat2 = np.deg2rad(lat2)
    long2 = np.deg2rad(long2)

    # A bunch of sub-calculations so that the calculation of s isn't a giant illegible nightmare
    sqLatDiff = (lat1 - lat2) ** 2
    sqLongDiff = (long1 - long2) ** 2
    cosArg = ((lat1) + (lat2)) / 2
    cosine = np.cos(cosArg) ** 2
    adjEarthRad = rho / 1e6 # Putting Earth's radius in the correct units (thousands of km)
    term1 = sqLongDiff * cosine
    
    # s
    s = adjEarthRad * np.sqrt(term1 + sqLatDiff)
    
    return s

def getErrorCorrelation(s: float) -> float:
    '''Calculate the background error correlation between two stations given the distance s between them, in thousands of km.'''
    expFactor = -1.24 
    
    exponent = expFactor * (s ** 2)
    out = 0.95 * np.exp(exponent)
    
    # Correction for two co-located observations
    if (s < 1e-8):
        out = 1

    return out


In [ ]:
### DO ANALYSIS ###

N = np.array([2,4,10]) # Number of stations to use for each analysis

def closestPoints(alat: float, along: float) -> np.ndarray:
    '''Take a point in lat/long and return an array of indices that will put any of our other similarly-structured arrays into distance order.
       The ordering will work for the observation point x/y and lat/long arrays, the observation value array, and the calculated background observation value array.'''
    distances = []

    for i in range(len(obsx)):
        dist = greatCircleDistance(alat, along, observations[i,1], observations[i,2])
        distances.append(dist)
    
    distances = np.array(distances)
    fixArr = np.argsort(distances)
    return fixArr

def getMatrices(lata: float, longa: float, nearLats: np.ndarray, nearLongs: np.ndarray):
    '''Take an analysis point in lat/long, as well as a list of observation points sorted by proximity, and calculate the correlation matrices needed by the weight function.'''

    # Initializing the matrices that will hold the background error correlations
    # analysisMatrix is the column vector involving the analysis gridpoint, bigMatrix is the big KxK matrix of correlations
    analysisMatrix = np.ndarray((len(nearLats),))
    bigMatrix = np.ndarray((len(nearLats),len(nearLats)))
    
    # Loop over however many observation points were passed in to populate the matrices
    for i in range(len(nearLats)):
        ptDist = greatCircleDistance(lata, longa, nearLats[i], nearLongs[i])
        covar = getErrorCorrelation(ptDist)
        analysisMatrix[i] = covar
        for j in range(len(nearLats)):
            ptDist = greatCircleDistance(nearLats[i], nearLongs[i], nearLats[j], nearLongs[j])
            covar = getErrorCorrelation(ptDist)
            bigMatrix[i][j] = covar

    return analysisMatrix, bigMatrix

def analysisFunc(index: float, jndex: float, weights: np.ndarray, order: np.ndarray) -> float:
    '''The function to calculate an actual analysis value. Uses index and jndex in place of literal x/y or lat/long so we can use a grid of background values, which is easier'''
    
    obsIncs = observations[order, 3] - bgObservations[order]

    # I use the dot product here because it's literally exactly what we want and the numpy function is probably a lot more stable than anything I'd write
    weightTerm = np.dot(weights, obsIncs)
    anVal = bgGrid[index][jndex] + weightTerm
     
    return anVal

# Initializing lists for the analyses produced by the N=2,4,10 passes
# I do all three at once and stick them into a larger array so I can do all the analyses in a single triple-nested loop, which avoids duplicated code
analysis2 = []
analysis4 = []
analysis10 = []
a = [analysis2, analysis4, analysis10]

# I create a linear array of analysis values and a gridded array at the same time, because the linear one is easier for calculation but I need the grid for plotting
graphZ2 = np.ndarray((len(x),len(y)))
graphZ4 = np.ndarray((len(x),len(y)))
graphZ10 = np.ndarray((len(x),len(y)))
z = [graphZ2, graphZ4, graphZ10]

# Lists to contain the forward-(int)erpolated (An)alyses at observation locations, calculated using the analyses once those are themselves calculated
intAn2 = []
intAn4 = []
intAn10 = []
intAns = [intAn2, intAn4, intAn10]

for n in range(len(N)):
    for i in range(len(x)):
        for j in range(len(y)):
            # Immediately turn (x,y) into lat/long, and then get an array of indices in the order that would sort an array of points into distance order  
            # At the same time, truncate the lists to the desired length. This dramatically shrinks the matrix that will need to be inverted, reducing runtime by several orders of magnitude (i tested) 
            lat = analysisLat[i, j]
            lon = analysisLong[i, j]
            order = closestPoints(lat, lon)[:N[n]]
            nLats = observations[order, 1]
            nLongs = observations[order, 2]

            # Get correlation arrays
            anMat, bigMat = getMatrices(lat, lon, nLats, nLongs )
            
            # The built-in numpy function should be more numerically stable than manual inversion
            # I don't think manual inversion is required?? And I demonstrated that I can do it in the last hw anyway
            weights = np.linalg.solve(bigMat, anMat)

            # Find analysis value and populate analysis arrays
            currVal = analysisFunc(i, j, weights, order)
            a[n].append(currVal)
            z[n][i][j] = currVal

    # Convert lists to arrays because arrays stay winning
    a[n] = np.array(a[n])

    # We quickly forward-interpolate the analysis to observation locations so we can find analysis differences later
    interpAnalysis = backgroundFromAnalysis(a[n], z[n], bgObservations, roi)
    intAns[n] = interpAnalysis


In [ ]:
### ANALYSIS DIFFERENCES ###

# Initializing lists that will contain the differences between foward-interpolated observation values and real observations
diff2 = []
diff4 = []
diff10 = []
diffs = [diff2, diff4, diff10]

# Populate arrays by subtracting the observation values from the interpolated values at the observation locations
for n in range(len(N)):
   diffs[n] = intAns[n] - observations[:,3]
   diffs[n] = np.array(diffs[n])


In [ ]:
### RMS ###

rms = []
for n in range(len(N)):
    # We have to do the astype(float) thing so that we can use np.isfinite later
    square = np.square(intAns[n] - observations[:,3]).astype(float)
    
    # This variable is currently just the number of observations
    # However, it will soon tell us how many non-nan increment values there are so that we can accurately calculate the RMSs
    num = len(intAns[n])

    # First, check to see if there are any nans. If not, we can skip the loop and save some time
    if not np.all(np.isfinite(square)):
        for s in square:
            # If an increment is nan, we discard that observation
            # We don't need to actually remove it from the array because we'll use np.nansum instead
            if not np.isfinite(s):
                num -= 1
    rootMeanSquare = np.sqrt(np.nansum(square) / num)
    rms.append(rootMeanSquare)
rms = np.array(rms)

# Print the RMS values neatly :)
for r in range(len(rms)):
    print(f'RMS N = {N[r]}: ', rms[r])

In [ ]:
### INCREMENT FIELDS ###

# Create and populate arrays of the analysis increments (the differences between the different analyses)
inc42 =  graphZ4 - graphZ2
inc102 = graphZ10 - graphZ2
inc104 = graphZ10 - graphZ4

In [ ]:
### Plot 500mb analyses over a map ###

# CartoPy throws a fit if I don't include this
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

globe = ccrs.Globe(ellipse=None, semimajor_axis=rho/100, semiminor_axis=rho/100)
proj = ccrs.Stereographic(central_longitude=-100,central_latitude=90,true_scale_latitude=60,globe=globe)
fig = plt.figure(figsize=(8,8),dpi=200)
ax1 = fig.add_subplot(111,projection=proj)
ax1.add_feature(cfeature.STATES)
ax1.add_feature(cfeature.COASTLINE)
# plot analysis (MAY NEED TO CHANGE VARIABLE NAMES/INDICES) #
cs1 = ax1.contour(gridy/100/m,-gridx/100/m,graphZ10.T,colors='k',levels=np.arange(0,8000,60),transform=proj)
plt.clabel(cs1,levels=np.arange(0,8000,60))

ax1.set(title='500hPa Geopotential Heights (M) -- N = 10')

plt.show()

In [ ]:
### Plot analysis increments over a map ###

# CartoPy throws a fit if I don't include this
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

globe = ccrs.Globe(ellipse=None, semimajor_axis=rho/100, semiminor_axis=rho/100)
proj = ccrs.Stereographic(central_longitude=-100,central_latitude=90,true_scale_latitude=60,globe=globe)
fig = plt.figure(figsize=(8,8),dpi=200)
ax1 = fig.add_subplot(111,projection=proj)
ax1.add_feature(cfeature.STATES)
ax1.add_feature(cfeature.COASTLINE)
# plot analysis (MAY NEED TO CHANGE VARIABLE NAMES/INDICES) #
cs1 = ax1.contour(gridy/100/m,-gridx/100/m,inc42.T,colors='k',levels=np.arange(0,100,5),transform=proj)
plt.clabel(cs1,levels=np.arange(0,100,5))

ax1.set(title='500hPa Geopotential Height Analysis Increment (M): 4 - 2')

plt.show()

In [ ]:
### In a separte text file (or below), answer the following questions ###
'''
1 - Describe the general features that you see in your contoured analyses.
    

2 - Describe the differences that you see in your contoured analyses.  
    Does one analysis seem to be smoother than the other?  If so, what would cause this?
    

3 - What happens as you increase the number of points considered for the analysis?  Is this 
    desirable?  Why or why not?
    
'''

1. As before, there appears to be a deep, tilted trough (negatively?? i still mix those up) running from central BC through the Rockies, ending up right around the TX-OK-NM border. I think the diverging isohypses over the Great Plains could be a cause of some storms? Because the upper-level divergence will suck air upwards from the surface, strengthening convection, right?

2. The contours definitely appear to smooth out with more observations. I'd imagine that, as we saw with successive corrections, taking a larger area into account smooths out instrument error and local noise, giving us a better picture of the synoptic arrangement.

3. Exactly what I said in the last question; more observations makes a smoother chart. You can see this clearly in the southern end of the 5460 hPa isohypse, and also over the east coast of Baja California. It is desirable, as it averages out instrument error and small-scale noise. I don't believe there's a downside to using an arbitrarily high number of stations, except that the program would really start to slow down. You'd be inverting a 135x135 matrix, which is already rough, but I'd guess that real-world analyses have far, far more datapoints.